# 🔍 Training YOLOv8 — Aksara Ulu Rejang

**Langkah:**
1. Pastikan GPU aktif: Runtime → Change runtime type → **T4 GPU**
2. Jalankan semua cell dari atas ke bawah
3. Download `best.pt` di cell terakhir
4. Taruh `best.pt` ke folder `kaganga-yolo/models/`

## Cell 1 — Cek GPU

In [ ]:
!nvidia-smi
import torch
print('GPU tersedia:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Tidak ada GPU')

## Cell 2 — Install Ultralytics

In [ ]:
!pip install ultralytics roboflow -q
print('Install selesai!')

## Cell 3 — Download Dataset dari Roboflow

Buka https://app.roboflow.com → Settings → Roboflow API → copy API Key kamu

In [ ]:
from roboflow import Roboflow

# ⚠️ GANTI dengan API key kamu dari https://app.roboflow.com/settings/api
API_KEY = 'GANTI_DENGAN_API_KEY_KAMU'

rf = Roboflow(api_key=API_KEY)
project = rf.workspace('novalrizkiansyah-ymail-com').project('aksara-ulu-rejang')
dataset = project.version(4).download('yolov8')

print('Dataset path:', dataset.location)
DATA_YAML = dataset.location + '/data.yaml'
print('data.yaml:', DATA_YAML)

## Cell 3B — ALTERNATIF: Upload ZIP dari PC

Gunakan cell ini JIKA tidak punya API key Roboflow.

Cara zip dataset di PC:
```
Klik kanan folder 'Aksara Ulu Rejang.v4-fix.yolov8' → Send to → Compressed (zip)
```

In [ ]:
# JALANKAN CELL INI HANYA JIKA TIDAK PAKAI ROBOFLOW API
# Hapus tanda # di bawah untuk mengaktifkan

# from google.colab import files
# import zipfile, os
#
# print('Pilih file ZIP dataset...')
# uploaded = files.upload()
# zip_name = list(uploaded.keys())[0]
#
# with zipfile.ZipFile(zip_name, 'r') as z:
#     z.extractall('/content/dataset')
#
# # Cari data.yaml
# for root, dirs, files_list in os.walk('/content/dataset'):
#     for f in files_list:
#         if f == 'data.yaml':
#             DATA_YAML = os.path.join(root, f)
#             print('data.yaml ditemukan:', DATA_YAML)
#             break

print('Cell ini dinonaktifkan. Aktifkan jika tidak pakai Roboflow API.')

## Cell 4 — Fix Path di data.yaml

In [ ]:
import yaml, os

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

# Tampilkan isi asli
print('Isi data.yaml asli:')
print(f"  train: {data.get('train')}")
print(f"  val:   {data.get('val')}")
print(f"  nc:    {data.get('nc')}")
print(f"  names sample: {data.get('names', [])[:5]}")

# Fix path ke absolute
base = os.path.dirname(DATA_YAML)
data['train'] = os.path.join(base, 'train', 'images')
data['val']   = os.path.join(base, 'valid', 'images')
data['test']  = os.path.join(base, 'test',  'images')

with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('\nPath sudah diupdate:')
print(f"  train: {data['train']}")
print(f"  val:   {data['val']}")
print(f"  Jumlah kelas: {data['nc']}")

## Cell 5 — Training YOLOv8

In [ ]:
from ultralytics import YOLO

# Pilih model:
# yolov8n.pt = nano  (paling cepat, akurasi sedang) ~5 menit
# yolov8s.pt = small (lebih akurat)                 ~10 menit
# yolov8m.pt = medium (akurasi bagus)               ~20 menit
MODEL = 'yolov8n.pt'

model = YOLO(MODEL)

results = model.train(
    data      = DATA_YAML,
    epochs    = 50,
    imgsz     = 640,
    batch     = 32,
    patience  = 10,
    device    = 0,        # GPU
    project   = '/content/runs',
    name      = 'aksara_ulu',
    exist_ok  = True,
    verbose   = True,
)

print('\n' + '='*50)
print('Training SELESAI!')
print(f"mAP@50:    {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP@50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
print('='*50)

## Cell 6 — Validasi Model

In [ ]:
# Load model terbaik dan validasi
best_model = YOLO('/content/runs/aksara_ulu/weights/best.pt')
metrics = best_model.val(data=DATA_YAML, device=0)

print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

## Cell 7 — Download Model ke PC

Setelah download, taruh `best.pt` ke:
```
D:\project\convis\kaganga\kaganga-yolo\models\best.pt
```

In [ ]:
from google.colab import files
import shutil

# Copy ke lokasi mudah diakses
shutil.copy('/content/runs/aksara_ulu/weights/best.pt', '/content/best.pt')

print('Mendownload best.pt...')
files.download('/content/best.pt')
print('Download selesai!')
print()
print('Sekarang taruh best.pt ke:')
print('D:\\project\\convis\\kaganga\\kaganga-yolo\\models\\best.pt')

## Cell 8 — (Opsional) Simpan ke Google Drive

In [ ]:
# Simpan ke Google Drive supaya tidak hilang kalau session mati
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(
    '/content/runs/aksara_ulu/weights/best.pt',
    '/content/drive/MyDrive/aksara_ulu_best.pt'
)
print('Tersimpan di Google Drive: aksara_ulu_best.pt')